In [1]:
import scanpy as sc

/Users/chrislangseth/miniforge3/envs/cellcharter/lib/python3.11/site-packages/louvain/__init__.py:54: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution, DistributionNotFound


In [2]:
ad = sc.read_h5ad('/Users/chrislangseth/Downloads/merged_lerma_martin.h5ad')
ad.obs_names_make_unique()
ad

/Users/chrislangseth/miniforge3/envs/cellcharter/lib/python3.11/site-packages/anndata/_core/anndata.py:1806: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


AnnData object with n_obs × n_vars = 67851 × 23049
    obs: 'array_row', 'array_col', 'patient_id', 'sample_id', 'condition', 'lesion_type', 'age', 'sex', 'rin', 'pmi_hrs', 'duration_y', 'ms_class', 'cause_death', 'batch_vs', 'areas', 'niches'
    obsm: 'ccc_scores', 'ctype_abunds', 'ctype_props', 'hallmark_scores', 'progeny_scores', 'reactome_scores', 'spatial'

In [3]:
ad.var['mt'] = ad.var_names.str.upper().str.startswith('MT-')
sc.pp.calculate_qc_metrics(ad, qc_vars=['mt'], inplace=True)
ad.obs[['total_counts', 'n_genes_by_counts', 'pct_counts_mt']].describe()

,total_counts,n_genes_by_counts,pct_counts_mt
count,67851.000000,67851.000000,67851.000000
mean,3055.793945,1373.133233,16.654997
std,2776.642334,965.027084,14.188172
min,239.000000,199.000000,0.837776
25%,1179.000000,675.000000,8.993136
50%,2223.000000,1126.000000,12.947300
75%,3990.000000,1795.000000,18.498800
max,33544.000000,7289.000000,87.945450


In [4]:
ad.layers['counts'] = ad.X.copy()
ad = ad[:, ~ad.var['mt']].copy()
ad

AnnData object with n_obs × n_vars = 67851 × 23036
    obs: 'array_row', 'array_col', 'patient_id', 'sample_id', 'condition', 'lesion_type', 'age', 'sex', 'rin', 'pmi_hrs', 'duration_y', 'ms_class', 'cause_death', 'batch_vs', 'areas', 'niches', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt'
    var: 'mt', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts'
    obsm: 'ccc_scores', 'ctype_abunds', 'ctype_props', 'hallmark_scores', 'progeny_scores', 'reactome_scores', 'spatial'
    layers: 'counts'

In [10]:
sc.pp.normalize_total(ad, target_sum=1e4)
sc.pp.log1p(ad)
ad.raw = ad
ad

AnnData object with n_obs × n_vars = 67851 × 23036
    obs: 'array_row', 'array_col', 'patient_id', 'sample_id', 'condition', 'lesion_type', 'age', 'sex', 'rin', 'pmi_hrs', 'duration_y', 'ms_class', 'cause_death', 'batch_vs', 'areas', 'niches', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt'
    var: 'mt', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts'
    uns: 'log1p'
    obsm: 'ccc_scores', 'ctype_abunds', 'ctype_props', 'hallmark_scores', 'progeny_scores', 'reactome_scores', 'spatial'
    layers: 'counts'

In [11]:
ad.X.max()

np.float32(7.990899)

In [5]:
ad.write('/Users/chrislangseth/Downloads/merged_lerma_martin_processed.h5ad')

In [14]:
import numpy as np
from scipy import sparse


In [15]:
x = ad.layers["counts"]

vals = x.data if sparse.issparse(x) else np.asarray(x).ravel()
vals = vals[np.isfinite(vals)]

print("fraction non-integer:", np.mean(np.abs(vals - np.round(vals)) > 1e-6))
print("min/max:", vals.min(), vals.max())

row_sums = np.asarray(x.sum(axis=1)).ravel()
print("row sums:", row_sums.min(), np.median(row_sums), row_sums.max())


fraction non-integer: 0.0
min/max: 1.0 3547.0
row sums: 201.0 1793.0 29819.0


In [16]:
x = ad.X

vals = x.data if sparse.issparse(x) else np.asarray(x).ravel()
vals = vals[np.isfinite(vals)]

print("fraction non-integer:", np.mean(np.abs(vals - np.round(vals)) > 1e-6))
print("min/max:", vals.min(), vals.max())

row_sums = np.asarray(x.sum(axis=1)).ravel()
print("row sums:", row_sums.min(), np.median(row_sums), row_sums.max())


fraction non-integer: 1.0
min/max: 0.2891984 7.990899
row sums: 653.83826 2355.2947 5088.707
